<a href="https://colab.research.google.com/github/rodrigoiyg27/TopicosEspeciales2026/blob/main/EjemploClase7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# CSV oficial del paquete palmerpenguins (licencia CC0)
URL_PINGUINOS = (
    "https://raw.githubusercontent.com/allisonhorst/"
    "palmerpenguins/main/inst/extdata/penguins.csv"
)


pinguinos = pd.read_csv(URL_PINGUINOS)

# Verificación mínima: dimensiones esperadas (344 filas, 8 columnas)
print(pinguinos.shape)

(344, 8)


 ¿qué características explican las diferencias de masa corporal entre los pingüinos?

In [ ]:
print(pinguinos.shape)        # (filas, columnas)
pinguinos.info()              # tipo de cada columna y cuántos valores no nulos tiene
print(pinguinos.head(3))      # primeras filas: ¿el encabezado se leyó bien?
print(pinguinos.tail(3))      # últimas filas: ¿hay totales o basura al final?

(344, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 21.6+ KB
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   

   body_mass_g     sex  year  
0       3750.0  

Calidad de los datos

In [ ]:
nulos_por_columna = pinguinos.isna().sum()
# Columnas con mediciones numéricas;
columnas_numericas = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"
]
# Una fila por columna del dataset: tipo, nulos y cardinalidad
reporte_calidad = pd.DataFrame({
    "tipo": pinguinos.dtypes.astype(str),
    "nulos": nulos_por_columna,
    "pct_nulos": (nulos_por_columna / len(pinguinos) * 100).round(2),
    "valores_unicos": pinguinos.nunique(),
})
print(reporte_calidad)

# Filas idénticas en todas sus columnas
print("Filas duplicadas:", pinguinos.duplicated().sum())

# ¿Los nulos se concentran en algún grupo? any(axis=1) marca filas con al menos un nulo
filas_con_nulos = pinguinos[pinguinos.isna().any(axis=1)]
print(filas_con_nulos.groupby("species").size())

# Rangos plausibles: no debe haber medidas negativas ni en cero
print(pinguinos[columnas_numericas].agg(["min", "max"]))

                      tipo  nulos  pct_nulos  valores_unicos
species             object      0       0.00               3
island              object      0       0.00               3
bill_length_mm     float64      2       0.58             164
bill_depth_mm      float64      2       0.58              80
flipper_length_mm  float64      2       0.58              55
body_mass_g        float64      2       0.58              94
sex                 object     11       3.20               2
year                 int64      0       0.00               3
Filas duplicadas: 0
species
Adelie    6
Gentoo    5
dtype: int64
     bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
min            32.1           13.1              172.0       2700.0
max            59.6           21.5              231.0       6300.0


Análisis de variables de forma individual

```
# Esto tiene formato de código
```



In [ ]:
# Variables categóricas: frecuencia absoluta y relativa, incluyendo nulos
for columna in ["species", "island", "sex"]:
    conteo = pinguinos[columna].value_counts(dropna=False)
    porcentaje = (conteo / len(pinguinos) * 100).round(1)
    print(pd.DataFrame({"n": conteo, "pct": porcentaje}), "\n")

# Variables numéricas: centro, dispersión y forma en una sola tabla
resumen_numerico = (
    pinguinos[columnas_numericas]
    .agg(["mean", "median", "std", "skew"])
    .T                                    # transpone: una fila por variable
)
resumen_numerico["cv_pct"] = resumen_numerico["std"] / resumen_numerico["mean"] * 100
print(resumen_numerico.round(2))

             n   pct
species             
Adelie     152  44.2
Gentoo     124  36.0
Chinstrap   68  19.8 

             n   pct
island              
Biscoe     168  48.8
Dream      124  36.0
Torgersen   52  15.1 

          n   pct
sex              
male    168  48.8
female  165  48.0
NaN      11   3.2 

                      mean   median     std  skew  cv_pct
bill_length_mm       43.92    44.45    5.46  0.05   12.43
bill_depth_mm        17.15    17.30    1.97 -0.14   11.51
flipper_length_mm   200.92   197.00   14.06  0.35    7.00
body_mass_g        4201.75  4050.00  801.95  0.47   19.09


Hallazgos del paso #4


1.   Las especies no están balanceadas los Adelie son el 44.2 % y los Chinstrap apenas el 19.8 %. Cualquier promedio global va a parecerse más a los Adelie, que pesan más en la mezcla.
2.   Lan isla Biscoe tiene el 48.8% de los habitantes y Torgesen solo el 15.1%


1.   168 machos y 164 hembras








Proporción de las dimensiones de los picos


In [ ]:
# Media del largo y del alto del pico dentro de cada especie
pico_por_especie = (
    pinguinos
    .groupby("species")[["bill_length_mm", "bill_depth_mm"]]
    .mean()
    .round(1)
)
print(pico_por_especie)

           bill_length_mm  bill_depth_mm
species                                 
Adelie               38.8           18.3
Chinstrap            48.8           18.4
Gentoo               47.5           15.0


Relaciones entre variables

In [ ]:
# ¿Cómo se reparten las especies entre las islas?
print(pd.crosstab(pinguinos["species"], pinguinos["island"]))

# Masa mediana por isla
print("\n \n Masa mediana por isla")
print(pinguinos.groupby("island")["body_mass_g"].median())

# ...Adelie. La única presente en 3 islas
adelie = pinguinos[pinguinos["species"] == "Adelie"]
print("\n \n Solo Especie Adelie")
print(adelie.groupby("island")["body_mass_g"].agg(n="count", mediana="median"))

# Masa mediana por especie y sexo, con la diferencia entre sexos
print("\n \n Masa mediana por especie y sexo")
masa_por_especie_y_sexo = pinguinos.pivot_table(
    index="species", columns="sex", values="body_mass_g", aggfunc="median"
)
masa_por_especie_y_sexo["diferencia_g"] = (
    masa_por_especie_y_sexo["male"] - masa_por_especie_y_sexo["female"]
)
masa_por_especie_y_sexo["diferencia_pct"] = (
    masa_por_especie_y_sexo["diferencia_g"] / masa_por_especie_y_sexo["female"] * 100
).round(1)
print(masa_por_especie_y_sexo)

island     Biscoe  Dream  Torgersen
species                            
Adelie         44     56         52
Chinstrap       0     68          0
Gentoo        124      0          0

 
 Masa mediana por isla
island
Biscoe       4775.0
Dream        3687.5
Torgersen    3700.0
Name: body_mass_g, dtype: float64

 
 Solo Especie Adelie
            n  mediana
island                
Biscoe     44   3750.0
Dream      56   3575.0
Torgersen  51   3700.0

 
 Masa mediana por especie y sexo
sex        female    male  diferencia_g  diferencia_pct
species                                                
Adelie     3400.0  4000.0         600.0            17.6
Chinstrap  3550.0  3950.0         400.0            11.3
Gentoo     4700.0  5500.0         800.0            17.0


Analisis de Hallazgos

In [ ]:
def correlacion_por_grupo(datos: pd.DataFrame, grupo: str, x: str, y: str) -> pd.Series:
    """Correlación de Pearson entre x e y dentro de cada categoría de `grupo`."""
    return pd.Series({
        categoria: subconjunto[x].corr(subconjunto[y])
        for categoria, subconjunto in datos.groupby(grupo)
    }).round(3)

# La correlación global aleta-masa es 0.871. ¿Se mantiene dentro de cada especie?
print("\n Correlación aleta-masa")
print(correlacion_por_grupo(pinguinos, "species", "flipper_length_mm", "body_mass_g"))

# La masa mediana sube en 2008. ¿Engordaron, o cambió la mezcla de especies?
print("\n Masa mediana por año")
print(pinguinos.groupby("year")["body_mass_g"].median())
print("\n Pinguinos por año y especies")
print(pd.crosstab(pinguinos["year"], pinguinos["species"]))
print("\n masa mediana por año y especie")
print(
    pinguinos.pivot_table(
        index="year", columns="species", values="body_mass_g", aggfunc="median"
    )
)


 Correlación aleta-masa
Adelie       0.468
Chinstrap    0.642
Gentoo       0.703
dtype: float64

 Masa mediana por año
year
2007    3900.0
2008    4200.0
2009    4000.0
Name: body_mass_g, dtype: float64

 Pinguinos por año y especies
species  Adelie  Chinstrap  Gentoo
year                              
2007         50         26      34
2008         50         18      46
2009         52         24      44

 masa mediana por año y especie
species  Adelie  Chinstrap  Gentoo
year                              
2007     3650.0     3700.0  5050.0
2008     3700.0     3750.0  5000.0
2009     3600.0     3675.0  5000.0


In [ ]:
# Correlación global entre largo y alto del pico
print("\n Correlación global entre largo y alto del pico")
r_pico_global = pinguinos["bill_length_mm"].corr(pinguinos["bill_depth_mm"])
print(f"Global: {r_pico_global:.3f}")
print("\n Correlación por especie")
print(correlacion_por_grupo(pinguinos, "species", "bill_length_mm", "bill_depth_mm"))


 Correlación global entre largo y alto del pico
Global: -0.235

 Correlación por especie
Adelie       0.391
Chinstrap    0.654
Gentoo       0.643
dtype: float64
